In [1]:
from pathlib import Path
from tabulate import tabulate

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

In [2]:
project_dir = Path("K:/PythonWork/downscaling/Kaya_downscaling")

In [5]:
profiles = ["base_run", "sensitivity_3", "sensitivity_4", "sensitivity_5", "sensitivity_Zhuang"]
dir = Path(f"K:/PythonWork/downscaling/Kaya_downscaling/data/processed")

#scenario_run = "IMAGE_ScenarioMIP_IMAGE 3.4_Medium - SSP2_conv_year_2150_net"
scenario_run = "IMAGE_ScenarioMIP_IMAGE 3.4_Medium - SSP2_conv_year_2150_net"
em_file = "Emissions_CO2_Excl_shipping_aviation_AFOLU_unharmonised_SSP2.nc"

In [10]:
xr_emissions = xr.open_dataset(Path(dir / profiles[0] / scenario_run) / em_file)
xr_emissions

<xarray.Dataset> Size: 622MB
Dimensions:                                     (time: 12, y: 1800, x: 3600)
Coordinates:
  * x                                           (x) float64 29kB -179.9 ... 1...
  * y                                           (y) float64 14kB 89.95 ... -8...
  * time                                        (time) int64 96B 2020 ... 2100
    spatial_ref                                 int64 8B ...
Data variables:
    Emissions_CO2_Excl_shipping_aviation_AFOLU  (time, y, x) float64 622MB ...

In [11]:
# with pathlib find emfile in the directoy profile (from profiles) and scenario_run, open it, make it into a dataframe, and collect it into a general dataframe
df_total_runs = pd.DataFrame()
for profile in profiles:
    em_dir = Path(dir / profile / scenario_run)
    print(f"Processing {em_dir / em_file}")

    xr_emissions = xr.open_dataset(em_dir / em_file)
    global_per_year = xr_emissions["Emissions_CO2_Excl_shipping_aviation_AFOLU"].sum(dim=["y", "x"])
    df_em = global_per_year.to_dataframe().reset_index()

    df_em.drop(columns=["spatial_ref"], inplace=True, errors="ignore")
    df_em["run"] = f"{profile}_{scenario_run}"
    df_em.rename(columns={"time": "year"}, inplace=True)

    df_total_runs = pd.concat([df_total_runs, df_em])

df_total_runs

Processing K:\PythonWork\downscaling\Kaya_downscaling\data\processed\base_run\IMAGE_ScenarioMIP_IMAGE 3.4_Medium - SSP2_conv_year_2150_net\Emissions_CO2_Excl_shipping_aviation_AFOLU_unharmonised_SSP2.nc
Processing K:\PythonWork\downscaling\Kaya_downscaling\data\processed\sensitivity_3\IMAGE_ScenarioMIP_IMAGE 3.4_Medium - SSP2_conv_year_2150_net\Emissions_CO2_Excl_shipping_aviation_AFOLU_unharmonised_SSP2.nc
Processing K:\PythonWork\downscaling\Kaya_downscaling\data\processed\sensitivity_4\IMAGE_ScenarioMIP_IMAGE 3.4_Medium - SSP2_conv_year_2150_net\Emissions_CO2_Excl_shipping_aviation_AFOLU_unharmonised_SSP2.nc
Processing K:\PythonWork\downscaling\Kaya_downscaling\data\processed\sensitivity_5\IMAGE_ScenarioMIP_IMAGE 3.4_Medium - SSP2_conv_year_2150_net\Emissions_CO2_Excl_shipping_aviation_AFOLU_unharmonised_SSP2.nc
Processing K:\PythonWork\downscaling\Kaya_downscaling\data\processed\sensitivity_Zhuang\IMAGE_ScenarioMIP_IMAGE 3.4_Medium - SSP2_conv_year_2150_net\Emissions_CO2_Excl_shipp

,year,Emissions_CO2_Excl_shipping_aviation_AFOLU,run
0,2020,3.363636e+10,base_run_IMAGE_ScenarioMIP_IMAGE 3.4_Medium - ...
1,2025,3.671853e+10,base_run_IMAGE_ScenarioMIP_IMAGE 3.4_Medium - ...
2,2030,3.629789e+10,base_run_IMAGE_ScenarioMIP_IMAGE 3.4_Medium - ...
3,2035,3.498588e+10,base_run_IMAGE_ScenarioMIP_IMAGE 3.4_Medium - ...
4,2040,3.357318e+10,base_run_IMAGE_ScenarioMIP_IMAGE 3.4_Medium - ...
5,2045,3.151457e+10,base_run_IMAGE_ScenarioMIP_IMAGE 3.4_Medium - ...
6,2050,2.918218e+10,base_run_IMAGE_ScenarioMIP_IMAGE 3.4_Medium - ...
7,2060,2.672434e+10,base_run_IMAGE_ScenarioMIP_IMAGE 3.4_Medium - ...
8,2070,2.511976e+10,base_run_IMAGE_ScenarioMIP_IMAGE 3.4_Medium - ...
9,2080,2.430872e+10,base_run_IMAGE_ScenarioMIP_IMAGE 3.4_Medium - ...


In [12]:
df_print = df_total_runs.pivot(index="year", columns="run", values="Emissions_CO2_Excl_shipping_aviation_AFOLU")
#print(tabulate(df_print, headers="keys", tablefmt="grid", showindex=False, floatfmt=",.1f", intfmt=","))
df_print.to_csv(project_dir / "data" / "check" / f"check_results_{scenario_run}.csv", index=True, float_format="%.1f", sep=";")